In [3]:
import numpy as np
import pandas as pd
import pulp
from sensitivity import heuristic3
import json
from pathlib import Path
import torch

In [27]:
df = pd.read_csv("parameter_summary_with_rmse.csv")
tradeoff_map = (
    df[df["file"].str.contains("left_wall_dist")]
    [["rmse", "total_params"]]
    .to_numpy()
)
tradeoff_map = np.array([
    [0.024842333333333335,     139],
    [0.022418333333333333,     285],
    [0.005855666666666667,   34405],
    [0.008697000000000000,   68741],
    [0.001205666666666667,  137777],
    [0.001235666666666667,  275393],
    [0.001804333333333333,  550753],
])

In [28]:
sensitivity = [5.829618417354863, 5.829618417354863, 6.783432564382072]

In [29]:
selection = heuristic3(
    sensitivity=sensitivity,
    budget=100000,
    tradeoff_map=tradeoff_map,
)

print(selection)

[1, 1, 2]


In [10]:
jsonl_path = Path("./output-dir/dfac09_2/metrics.jsonl")

results = []

with open(jsonl_path, "r") as f:
    for line in f:
        row = json.loads(line)

        runs = row["runs"]

        baseline_rmse = None
        arch8_rmse = None

        for run in runs:
            if run["label"] == "comb1":
                baseline_rmse = run["rmse"]
            else:
                # assumes the other entry is your arch8 run
                arch8_rmse = run["rmse"]

        results.append({
            "map": row["map"],
            "baseline_rmse": baseline_rmse,
            "arch8_rmse": arch8_rmse,
        })


df = pd.DataFrame(results)

        

df.loc[len(df)] = {
    "map": "MEAN",
    "baseline_rmse": df["baseline_rmse"].mean(),
    "arch8_rmse": df["arch8_rmse"].mean(),
            }

df

,map,baseline_rmse,arch8_rmse
0,F1/MexicoCity/MexicoCity,0.136200,0.158300
1,F1/Monza/Monza,0.094400,0.114600
2,F1/Nuerburgring/Nuerburgring,0.144500,0.146200
3,F1/Shanghai/Shanghai,0.163400,0.148200
4,F1/Silverstone/Silverstone,0.129600,0.139100
5,F1/Sochi/Sochi,0.126600,0.139300
6,F1/Spa/Spa,0.115800,0.132500
7,MEAN,0.130071,0.139743


In [8]:


selection = [2, 2, 4]

# heuristic3 returns level indices, convert to arch numbers
track_arch = selection[0] + 1
left_arch = selection[1] + 1
heading_arch = selection[2] + 1

# --------------------------------------------------
# Arch1-7 parameter lookup table
# --------------------------------------------------

arch_sizes = {
    1: 139,
    2: 285,
    3: 34405,
    4: 68741,
    5: 137777,
    6: 275393,
    7: 550753,
}

baseline_sizes = {
    "track_width": arch_sizes[track_arch],
    "left_wall_dist": arch_sizes[left_arch],
    "heading_error": arch_sizes[heading_arch],
}

# --------------------------------------------------
# Arch8 models
# --------------------------------------------------

track_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/dfac09/track_width_arch8_trial101.pt"

left_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/dfac09/left_wall_dist_arch8_trial101.pt"

heading_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/dfac09/heading_error_arch8_trial101.pt"

def count_params(path):
    model = torch.jit.load(path, map_location="cpu")
    return sum(p.numel() for p in model.parameters())

arch8_sizes = {
    "track_width": count_params(track_path),
    "left_wall_dist": count_params(left_path),
    "heading_error": count_params(heading_path),
}

# --------------------------------------------------
# Build comparison table
# --------------------------------------------------

df = pd.DataFrame({
    "target": ["track_width", "left_wall_dist", "heading_error"],
    "baseline_arch": [track_arch, left_arch, heading_arch],
    "baseline_params": [
        baseline_sizes["track_width"],
        baseline_sizes["left_wall_dist"],
        baseline_sizes["heading_error"],
    ],
    "arch8_params": [
        arch8_sizes["track_width"],
        arch8_sizes["left_wall_dist"],
        arch8_sizes["heading_error"],
    ],
})

df["param_difference"] = (
    df["arch8_params"] - df["baseline_params"]
)

# Total row
df.loc[len(df)] = {
    "target": "TOTAL",
    "baseline_arch": "",
    "baseline_params": df["baseline_params"].sum(),
    "arch8_params": df["arch8_params"].sum(),
    "param_difference":
        df["arch8_params"].sum()
        - df["baseline_params"].sum(),
}

df

,target,baseline_arch,baseline_params,arch8_params,param_difference
0,track_width,3,34405,23249,-11156
1,left_wall_dist,3,34405,35069,664
2,heading_error,5,137777,137637,-140
3,TOTAL,,206587,195955,-10632
